In [6]:
import random

# ==========================================
# 1. MATRIZ DE LATÊNCIAS FÍSICAS D (10x10)
# ==========================================
D = [
    [0.0, 12.5, 22.0, 45.0, 31.0, 50.0, 62.0, 18.0, 40.0, 35.0],
    [12.5, 0.0, 15.0, 38.0, 25.0, 44.0, 55.0, 22.0, 35.0, 29.0],
    [22.0, 15.0, 0.0, 20.0, 18.0, 30.0, 42.0, 32.0, 25.0, 19.0],
    [45.0, 38.0, 20.0, 0.0, 14.0, 22.0, 35.0, 50.0, 15.0, 12.0],
    [31.0, 25.0, 18.0, 14.0, 0.0, 16.0, 28.0, 38.0, 12.0, 10.0],
    [50.0, 44.0, 30.0, 22.0, 16.0, 0.0, 15.0, 55.0, 20.0, 18.0],
    [62.0, 55.0, 42.0, 35.0, 28.0, 15.0, 0.0, 68.0, 32.0, 25.0],
    [18.0, 22.0, 32.0, 50.0, 38.0, 55.0, 68.0, 0.0, 45.0, 42.0],
    [40.0, 35.0, 25.0, 15.0, 12.0, 20.0, 32.0, 45.0, 0.0, 8.0],
    [35.0, 29.0, 19.0, 12.0, 10.0, 18.0, 25.0, 42.0, 8.0, 0.0]
]
NUM_NODES = 10

# ==========================================
# 2. CLASSE ACO PARA TOPOLOGIA DE REDE
# ==========================================
class NetworkACO:
    def __init__(self, num_ants=20, max_iter=50, rho=0.2, alpha=1.0, beta=2.0):
        self.num_ants = num_ants
        self.max_iter = max_iter
        self.rho = rho          # Taxa de evaporação (0.2)
        self.alpha = alpha      # Peso do feromônio
        self.beta = beta        # Peso da heurística (visibilidade = 1/D)

        # Inicialização da matriz de feromônio tau_ij
        self.tau = [[1.0 if i != j else 0.0 for j in range(NUM_NODES)] for i in range(NUM_NODES)]

    def construct_spanning_tree(self):
        """Constrói uma árvore geradora sem ciclos usando Prim Probabilístico"""
        visited = {0}
        edges = []

        while len(visited) < NUM_NODES:
            candidates = []
            for u in visited:
                for v in range(NUM_NODES):
                    if v not in visited and D[u][v] > 0:
                        candidates.append((u, v, D[u][v]))

            if not candidates:
                break # Segurança contra grafos desconexos

            # Calcula probabilidades baseadas em feromônio e visibilidade
            probs = []
            for u, v, cost in candidates:
                t = self.tau[u][v] ** self.alpha
                h = (1.0 / cost) ** self.beta
                probs.append(t * h)

            total_prob = sum(probs)
            if total_prob == 0:
                chosen = random.choice(candidates)
            else:
                probs = [p / total_prob for p in probs]
                r = random.random()
                cumulative = 0.0
                chosen = candidates[-1]
                for idx, p in enumerate(probs):
                    cumulative += p
                    if r <= cumulative:
                        chosen = candidates[idx]
                        break

            u, v, cost = chosen
            visited.add(v)
            edges.append((u, v, cost))

        return edges

    def calculate_tree_cost(self, edges):
        return sum(cost for _, _, cost in edges)

    def optimize(self):
        best_overall_edges = None
        best_overall_cost = float('inf')

        for _ in range(self.max_iter):
            ant_solutions = []

            # Construção de soluções pelas formigas
            for _ in range(self.num_ants):
                edges = self.construct_spanning_tree()
                if len(edges) == NUM_NODES - 1:
                    cost = self.calculate_tree_cost(edges)
                    ant_solutions.append((edges, cost))
                    if cost < best_overall_cost:
                        best_overall_cost = cost
                        best_overall_edges = edges

            if not ant_solutions:
                continue

            # Evaporação global do feromônio (rho = 0.2)
            for i in range(NUM_NODES):
                for j in range(NUM_NODES):
                    if i != j:
                        self.tau[i][j] *= (1.0 - self.rho)

            # Atualização baseada apenas nas melhores topologias da iteração (Elitismo)
            ant_solutions.sort(key=lambda x: x[1])
            best_iter_edges, best_iter_cost = ant_solutions[0]

            q = 100.0
            for u, v, cost in best_iter_edges:
                deposit = q / best_iter_cost
                self.tau[u][v] += deposit
                self.tau[v][u] += deposit

        return best_overall_edges, best_overall_cost

def generate_random_tree():
    """Gera uma árvore geradora aleatória para baseline de comparação"""
    visited = {0}
    edges = []
    while len(visited) < NUM_NODES:
        u = random.choice(list(visited))
        v = random.choice([node for node in range(NUM_NODES) if node not in visited])
        visited.add(v)
        edges.append((u, v, D[u][v]))
    return edges

# ==========================================
# 3. EXIBIÇÃO DOS RESULTADOS E ARTEFATOS
# ==========================================
if __name__ == "__main__":
    random.seed(42)

    # Executa o ACO
    aco = NetworkACO(num_ants=25, max_iter=60, rho=0.2)
    best_edges, best_cost = aco.optimize()

    # Constrói a Matriz de Adjacência final (10x10) do grafo otimizado
    adj_matrix = [[0.0]*NUM_NODES for _ in range(NUM_NODES)]
    for u, v, cost in best_edges:
        adj_matrix[u][v] = cost
        adj_matrix[v][u] = cost

    # Baseline Aleatório
    random.seed(100)
    random_edges = generate_random_tree()
    random_cost = sum(c for _, _, c in random_edges)

    # Cálculo do Ganho Percentual
    gain_percentage = ((random_cost - best_cost) / random_cost) * 100.0

    print("\n" + "="*70)
    print("LAB 03 — RESULTADOS DA OTIMIZAÇÃO POR COLÔNIA DE FORMIGAS (ACO)")
    print("="*70)
    print(f"Custo Total (Latência Acumulada) - Topologia Aleatória: {random_cost:.2f} ms")
    print(f"Custo Total (Latência Acumulada) - Topologia Otimizada ACO: {best_cost:.2f} ms")
    print(f"Ganho de Redução de Latência: {gain_percentage:.2f}%")
    print("="*70)
    print("\nMatriz de Adjacência Final (10x10) do Grafo de Rede Otimizado:")
    print("-" * 70)
    print("   " + " ".join([f"{i:5}" for i in range(NUM_NODES)]))
    for i in range(NUM_NODES):
        row_str = f"{i}: " + " ".join([f"{adj_matrix[i][j]:5.1f}" for j in range(NUM_NODES)])
        print(row_str)
    print("="*70)


LAB 03 — RESULTADOS DA OTIMIZAÇÃO POR COLÔNIA DE FORMIGAS (ACO)
Custo Total (Latência Acumulada) - Topologia Aleatória: 297.00 ms
Custo Total (Latência Acumulada) - Topologia Otimizada ACO: 124.50 ms
Ganho de Redução de Latência: 58.08%

Matriz de Adjacência Final (10x10) do Grafo de Rede Otimizado:
----------------------------------------------------------------------
       0     1     2     3     4     5     6     7     8     9
0:   0.0  12.5   0.0   0.0   0.0   0.0   0.0  18.0   0.0   0.0
1:  12.5   0.0  15.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
2:   0.0  15.0   0.0   0.0  18.0   0.0   0.0   0.0   0.0   0.0
3:   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  12.0
4:   0.0   0.0  18.0   0.0   0.0  16.0   0.0   0.0   0.0  10.0
5:   0.0   0.0   0.0   0.0  16.0   0.0  15.0   0.0   0.0   0.0
6:   0.0   0.0   0.0   0.0   0.0  15.0   0.0   0.0   0.0   0.0
7:  18.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0
8:   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   8